# GFW threshold matching to ASTD
The notebook goes through the GFW matching of fishing vessels on ASTD

We compare the GFW to the ASTD fishing vessels paths for each month to find associated mmsi - shipid

## The comparison is done with data aggregated day by day, over 12 months

In [1]:
import pandas as pd
import os
import track_builder as tb
import numpy as np
import json
import plotly.graph_objects as go

gfw_2020_path = "D:/Stockage/GFW/mmsi-daily-csvs-10-v3-2020/"
astd_path = r"D:\Stockage\ASTD"
parquet_path = "../../examples/data/"

year = 2020
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

In [2]:
def load_data_gfw(parquet_file, source, year, months=None):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        data =  pd.read_parquet(parquet_file)
        print(f"Loaded data {parquet_file}, parameters ignored")
    else:
        dfs = []
        for file in os.listdir(source):
            if file.endswith(".csv") and str(year) in file:

                month = int(file.split("-")[6])

                if months is None or month in months:
                    print("Working with :", file)
                    path = os.path.join(source, file)
                    df = pd.read_csv(path)
                    dfs.append(df)

        if not dfs:
            raise ValueError("No matching CSV files found")

        data = pd.concat(dfs, ignore_index=True)
        data.to_parquet(parquet_file)
        print(f"Loaded data {parquet_file} from {source}")

    return data

def load_data(parquet_file, source, year, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        data =  pd.read_parquet(parquet_file)
        print(f"Loaded data {parquet_file}, parameters ignored")
    else:
        data = tb.load_astd_monthly(base_path=source, year=year, **kwargs)
        data.to_parquet(parquet_file)
        print(f"Loaded data {parquet_file} from {source}")

    return data

df_ASTD_2020 = load_data('all_segments2020.parquet', source=astd_path, year= year, months=months, remove_nan_rows="default", usecols="default", progress=True)
df_ASTD_2020 = df_ASTD_2020[df_ASTD_2020['astd_cat'].str.contains('Fishing vessels')].copy()
df_ASTD_2020["date_time_utc"] = pd.to_datetime(df_ASTD_2020["date_time_utc"])
display(df_ASTD_2020)
df_GFW_2020 = load_data_gfw('gfw_2020.parquet', source=gfw_2020_path, year = year, months = months)
df_GFW_2020['date'] = pd.to_datetime(df_GFW_2020['date'])
display(df_GFW_2020)

Loaded data ../examples/data/all_segments2020.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
0,3769,2020-01-01 00:00:00+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,5.356155,369,6.341843,62.431442
2,2877,2020-01-01 00:00:00+00:00,Iceland,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1.880442,19,-14.006558,65.263588
4,2989,2020-01-01 00:00:00+00:00,Lithuania,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,155.507141,43,8.763072,78.804428
8,2933,2020-01-01 00:00:02+00:00,Iceland,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,7.431257,260,-20.277617,63.446152
13,2339,2020-01-01 00:00:04+00:00,Russia,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,1.740952,474,18.997549,69.679932
...,...,...,...,...,...,...,...,...,...,...
43111541,2113,2020-12-31 23:59:51+00:00,Norway,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,993.163025,369,17.505766,74.173950
43111542,1634,2020-12-31 23:59:51+00:00,Russia,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,1627.817993,364,10.828934,77.686615
43111543,70,2020-12-31 23:59:51+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,0.551000,360,-22.718353,64.037857
43111546,1199,2020-12-31 23:59:54+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,0.849000,661,-22.425282,63.839603


Loaded data ../examples/data/gfw_2020.parquet, parameters ignored


,date,cell_ll_lat,cell_ll_lon,mmsi,hours,fishing_hours
0,2020-01-01,-74.1,-118.1,272645000,1.2505,1.2505
1,2020-01-01,-74.1,-118.3,272645000,2.4430,2.2005
2,2020-01-01,-74.1,-118.2,272645000,0.8663,0.8663
3,2020-01-01,-74.1,-118.0,272645000,1.3005,1.3005
4,2020-01-01,-74.1,-118.4,272645000,0.4086,0.0836
...,...,...,...,...,...,...
55991199,2020-12-31,79.2,8.8,273418680,0.8986,0.8986
55991200,2020-12-31,79.3,8.6,273352280,0.1138,0.1138
55991201,2020-12-31,79.3,8.4,273352280,0.8827,0.7363
55991202,2020-12-31,79.3,8.5,273352280,1.1450,0.9033


## Process GFW

In [3]:
# Define starting and ending time for the grid (max a month)
df_GFW_2020['month'] = df_GFW_2020['date'].dt.year.astype(str) + '-' + df_GFW_2020['date'].dt.month.astype(str).str.zfill(2)

# Converting grid origin to integer (this avoid float issues with imprecisions)
df_GFW_2020["grid_lon"] = (df_GFW_2020["cell_ll_lon"] * 10).astype('int16')
df_GFW_2020["grid_lat"] = (df_GFW_2020["cell_ll_lat"] * 10).astype('int16')

df_GFW_2020

,date,cell_ll_lat,cell_ll_lon,mmsi,hours,fishing_hours,month,grid_lon,grid_lat
0,2020-01-01,-74.1,-118.1,272645000,1.2505,1.2505,2020-01,-1181,-741
1,2020-01-01,-74.1,-118.3,272645000,2.4430,2.2005,2020-01,-1183,-741
2,2020-01-01,-74.1,-118.2,272645000,0.8663,0.8663,2020-01,-1182,-741
3,2020-01-01,-74.1,-118.0,272645000,1.3005,1.3005,2020-01,-1180,-741
4,2020-01-01,-74.1,-118.4,272645000,0.4086,0.0836,2020-01,-1184,-741
...,...,...,...,...,...,...,...,...,...
55991199,2020-12-31,79.2,8.8,273418680,0.8986,0.8986,2020-12,88,792
55991200,2020-12-31,79.3,8.6,273352280,0.1138,0.1138,2020-12,86,793
55991201,2020-12-31,79.3,8.4,273352280,0.8827,0.7363,2020-12,84,793
55991202,2020-12-31,79.3,8.5,273352280,1.1450,0.9033,2020-12,85,793


We're aggregating the positions per day instead of keeping the time spent in each

In [4]:
# Aggregating the data by day and by positions
df_gfw = df_GFW_2020[['date', 'mmsi', 'grid_lat', 'grid_lon', 'month']].drop_duplicates().copy()
display(df_gfw)

,date,mmsi,grid_lat,grid_lon,month
0,2020-01-01,272645000,-741,-1181,2020-01
1,2020-01-01,272645000,-741,-1183,2020-01
2,2020-01-01,272645000,-741,-1182,2020-01
3,2020-01-01,272645000,-741,-1180,2020-01
4,2020-01-01,272645000,-741,-1184,2020-01
...,...,...,...,...,...
55991199,2020-12-31,273418680,792,88,2020-12
55991200,2020-12-31,273352280,793,86,2020-12
55991201,2020-12-31,273352280,793,84,2020-12
55991202,2020-12-31,273352280,793,85,2020-12


In [5]:
# select_subset = False
#
# if select_subset:
#     # get gfw samples
#     gfw_work = (
#         df_gfw
#         .groupby(['mmsi', 'month'], group_keys=False)
#         .sample(frac=0.5, random_state=42)
#     )
#
#     gfw_work.sort_index(inplace=True)
#
# else:
#     # or keep full data
#     gfw_work = df_gfw.copy()

gfw_work = df_gfw.copy()

gfw_work

,date,mmsi,grid_lat,grid_lon,month
0,2020-01-01,272645000,-741,-1181,2020-01
1,2020-01-01,272645000,-741,-1183,2020-01
2,2020-01-01,272645000,-741,-1182,2020-01
3,2020-01-01,272645000,-741,-1180,2020-01
4,2020-01-01,272645000,-741,-1184,2020-01
...,...,...,...,...,...
55991199,2020-12-31,273418680,792,88,2020-12
55991200,2020-12-31,273352280,793,86,2020-12
55991201,2020-12-31,273352280,793,84,2020-12
55991202,2020-12-31,273352280,793,85,2020-12


## Process ASTD

In [6]:
df_ASTD_2020['month'] = df_ASTD_2020['date_time_utc'].dt.year.astype(str) + '-' + df_ASTD_2020['date_time_utc'].dt.month.astype(str).str.zfill(2)
df_ASTD_2020['date'] = pd.to_datetime(df_ASTD_2020['date_time_utc'].dt.date)

# Transform ASTD into grids to reduce compute overhead
df_ASTD_2020["grid_lon"] = np.floor(df_ASTD_2020["longitude"] * 10).astype('int16')
df_ASTD_2020["grid_lat"] = np.floor(df_ASTD_2020["latitude"] * 10).astype('int16')

df_ASTD_2020

,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,date,grid_lon,grid_lat
0,3769,2020-01-01 00:00:00+00:00,Norway,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,5.356155,369,6.341843,62.431442,2020-01,2020-01-01,63,624
2,2877,2020-01-01 00:00:00+00:00,Iceland,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1.880442,19,-14.006558,65.263588,2020-01,2020-01-01,-141,652
4,2989,2020-01-01 00:00:00+00:00,Lithuania,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,155.507141,43,8.763072,78.804428,2020-01,2020-01-01,87,788
8,2933,2020-01-01 00:00:02+00:00,Iceland,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,7.431257,260,-20.277617,63.446152,2020-01,2020-01-01,-203,634
13,2339,2020-01-01 00:00:04+00:00,Russia,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,1.740952,474,18.997549,69.679932,2020-01,2020-01-01,189,696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43111541,2113,2020-12-31 23:59:51+00:00,Norway,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,993.163025,369,17.505766,74.173950,2020-12,2020-12-31,175,741
43111542,1634,2020-12-31 23:59:51+00:00,Russia,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,1627.817993,364,10.828934,77.686615,2020-12,2020-12-31,108,776
43111543,70,2020-12-31 23:59:51+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,0.551000,360,-22.718353,64.037857,2020-12,2020-12-31,-228,640
43111546,1199,2020-12-31 23:59:54+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,0.849000,661,-22.425282,63.839603,2020-12,2020-12-31,-225,638


In [7]:
# Aggregating the data by day and by positions
df_astd = df_ASTD_2020[['date', 'shipid', 'grid_lat', 'grid_lon', 'month']].drop_duplicates().copy()
display(df_astd)

,date,shipid,grid_lat,grid_lon,month
0,2020-01-01,3769,624,63,2020-01
2,2020-01-01,2877,652,-141,2020-01
4,2020-01-01,2989,788,87,2020-01
8,2020-01-01,2933,634,-203,2020-01
13,2020-01-01,2339,696,189,2020-01
...,...,...,...,...,...
43111409,2020-12-31,1077,714,433,2020-12
43111426,2020-12-31,2309,774,105,2020-12
43111470,2020-12-31,2165,773,112,2020-12
43111503,2020-12-31,555,697,191,2020-12


## Merge (inner join) both by position and time

In [8]:
merged = gfw_work.merge(df_astd, on=["grid_lon", "grid_lat", "date", 'month'], how="inner")
merged

,date,mmsi,grid_lat,grid_lon,month,shipid
0,2020-01-01,303308000,476,-1224,2020-01,3924
1,2020-01-01,303308000,476,-1224,2020-01,1667
2,2020-01-01,303308000,476,-1224,2020-01,1470
3,2020-01-01,303308000,476,-1224,2020-01,1563
4,2020-01-01,303308000,476,-1224,2020-01,1718
...,...,...,...,...,...,...
5206759,2020-12-31,273418680,792,88,2020-12,1772
5206760,2020-12-31,273352280,793,86,2020-12,1772
5206761,2020-12-31,273352280,793,84,2020-12,1772
5206762,2020-12-31,273352280,793,85,2020-12,1772


## Get shipids per mmsi

In [18]:
# Number of matched points per shipid, per mmsi, per month
match_size = (
    merged
    .groupby(['mmsi', 'month', 'shipid'])
    .agg(match_n_mmsi=('mmsi', 'size'))
    .reset_index()
)

# Total ASTD number of points per shipid, per month
astd_total = (
    df_astd
    .groupby(['shipid', 'month'])
    .agg(astd_n_ship=('shipid', 'size'))
    .reset_index()
)

# Total selected GFW number of points per mmsi, per month
gfw_total = (
    df_gfw
    .groupby(['mmsi', 'month'])
    .agg(gfw_n_mmsi=('mmsi', 'size'))
    .reset_index()
)

# Merge all
merged_scores = (
    match_size
    .merge(astd_total, on=['shipid', 'month'], how='inner')
    .merge(gfw_total, on=['mmsi', 'month'], how='inner')
)

# Compute scores
merged_scores['ratio_gfw'] = merged_scores['match_n_mmsi'] / merged_scores['gfw_n_mmsi'] # Proportion of matched gfw points
merged_scores['ratio_astd'] = merged_scores['match_n_mmsi'] / merged_scores['astd_n_ship'] # Proportion of matched astd points

merged_scores['score'] = merged_scores['ratio_gfw'] + merged_scores['ratio_astd']
merged_scores['max_score'] = merged_scores.groupby(['mmsi', 'month'])['score'].transform('max')

# Thresholds
t_astd = 0.9
t_gfw_high = 0.1
t_gfw_low = 0.7
n_small = 10

merged_scores = merged_scores[
    ( # = allowed missing astd datapoints
        ((merged_scores['match_n_mmsi'] < n_small) & (merged_scores['ratio_gfw'] >= t_gfw_low)) | # Ratio has to bet_gfw_low with tracks with less than n_small points
        ((merged_scores['match_n_mmsi'] >= n_small) & (merged_scores['ratio_gfw'] >= t_gfw_high)) # Else, check if ration is higher than t_gfw_high
    )
    # = allowed missing gfw datapoints
    & (merged_scores['ratio_astd'] >= t_astd)
    & (merged_scores['score'] == merged_scores['max_score'])
]

merged_scores = merged_scores.drop(columns=['max_score'])
merged_scores = merged_scores.sort_values(by='gfw_n_mmsi', ascending=False).reset_index(drop=True)

merged_scores

,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score
0,257139000,2020-09,2758,1853,1871,2608,0.710506,0.990379,1.700886
1,273431690,2020-10,2240,1235,1258,1870,0.660428,0.981717,1.642145
2,273214810,2020-08,2608,973,999,1761,0.552527,0.973974,1.526501
3,257913000,2020-09,5821,1189,1295,1748,0.680206,0.918147,1.598353
4,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592
...,...,...,...,...,...,...,...,...,...
4510,251153000,2020-06,1557,3,3,3,1.000000,1.000000,2.000000
4511,257046460,2020-10,478,2,2,2,1.000000,1.000000,2.000000
4512,231189000,2020-05,22009,2,2,2,1.000000,1.000000,2.000000
4513,367603000,2020-11,3967,1,1,1,1.000000,1.000000,2.000000


In [27]:
df = merged_scores.copy()

# Remove multiple shipid for one mmsi (in case many have the same max score)
mask = df.groupby(['mmsi', 'month'])['shipid'].transform('nunique') == 1

# Remove multiple mmsi for one shipid
mask1 = df.groupby(['shipid', 'month'])['mmsi'].transform('nunique') == 1


df = df[mask1 & mask]
df = df.sort_values(by='match_n_mmsi', ascending=False).reset_index(drop=True)


print("Total size (unique shipid):", df.shape[0])
print("Total mmsi :", df['mmsi'].nunique())

df_pct = ( # Pourcentage, of how many mmsi have 12, 11, 10 ... matched months
    df
    .groupby('mmsi')['month']
    .nunique()
    .value_counts(normalize=True)
    .mul(100)
    .rename('percent')
    .reset_index(name='percent')
    .rename(columns={'index': 'n_months'})
)

display(df_pct)

df


Total size : 2807
Total mmsi : 348
Total shipid : 2807


,month,percent
0,12,27.298851
1,11,12.931034
2,9,8.908046
3,10,8.333333
4,1,7.183908
5,3,6.609195
6,2,6.034483
7,6,5.172414
8,4,4.885057
9,5,4.597701


,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score
0,257139000,2020-09,2758,1853,1871,2608,0.710506,0.990379,1.700886
1,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592
2,276806000,2020-10,641,1332,1355,1577,0.844642,0.983026,1.827668
3,258535000,2020-10,1219,1301,1370,1513,0.859881,0.949635,1.809516
4,259666000,2020-10,428,1290,1308,1617,0.797774,0.986239,1.784012
...,...,...,...,...,...,...,...,...,...
2802,273425250,2020-10,14711,3,3,4,0.750000,1.000000,1.750000
2803,316022149,2020-03,2965,3,3,4,0.750000,1.000000,1.750000
2804,251153000,2020-06,1557,3,3,3,1.000000,1.000000,2.000000
2805,257046460,2020-10,478,2,2,2,1.000000,1.000000,2.000000


In [35]:
# Convert to a dictionnary - makes it easier to get the shipid for each mmsi
result = {}

for _, row in df.iterrows():
    mmsi = row['mmsi']
    month = row['month']
    shipid = row['shipid']
    result.setdefault(mmsi, {})[month] = shipid


# Save dictionnary to json
# with open(parquet_path + "gfw_threshmatch.json", "w") as f:
#     json.dump(result, f, indent=4)

# Save into csv
df.to_csv(parquet_path + "gfw_threshmatch.csv", index=False)

## Display examples

In [32]:
mmsi = 231020000

display(gfw_work[gfw_work['mmsi']==mmsi])

,date,mmsi,grid_lat,grid_lon,month
101737,2020-01-01,231020000,621,-72,2020-01
223682,2020-01-02,231020000,621,-72,2020-01
349233,2020-01-03,231020000,600,-32,2020-01
349234,2020-01-03,231020000,600,-31,2020-01
349235,2020-01-03,231020000,600,-27,2020-01
...,...,...,...,...,...
55529900,2020-12-27,231020000,621,-72,2020-12
55668833,2020-12-28,231020000,621,-72,2020-12
55789134,2020-12-29,231020000,621,-72,2020-12
55884099,2020-12-30,231020000,621,-72,2020-12


In [33]:
top_ships = result[mmsi]
# mmsi=273418680
# top_ships = {'2020-02' : 2915}

top_ships_df = pd.DataFrame(list(top_ships.items()), columns=['month', 'shipid'])

col_lat_astd = 'grid_lat'
col_lon_astd = 'grid_lon'

col_lat_gfw = 'grid_lat'
col_lon_gfw = 'grid_lon'

# To plot astd origin point or grids + gfw sample or not
origin = True

if origin:

    col_lat_astd = 'latitude'
    col_lon_astd = 'longitude'

    # col_lat_gfw = 'cell_ll_lat'
    # col_lon_gfw = 'cell_ll_lon'
    df1 = df_ASTD_2020.merge(top_ships_df, on=['shipid', 'month'], how='inner')
    df1['track_id'] = mmsi
    df2 = df_GFW_2020[df_GFW_2020['mmsi']==mmsi].merge(top_ships_df, on=['month'], how='inner')

else:
    df2 = gfw_work[(gfw_work['mmsi']==mmsi)].merge(top_ships_df, on=['month'], how='inner')
    df1 = df_astd.merge(top_ships_df, on=['shipid', 'month'], how='inner')

df2 = df2.drop_duplicates(subset=[col_lon_gfw, col_lat_gfw])

display(df1)
display(df2)

,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,date,grid_lon,grid_lat,track_id
0,1845,2020-01-01 00:05:19+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,3.520215,700,-7.170667,62.151779,2020-01,2020-01-01,-72,621,231020000
1,1845,2020-01-01 00:17:00+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,5.923629,380,-7.170733,62.151775,2020-01,2020-01-01,-72,621,231020000
2,1845,2020-01-01 00:23:19+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2.573171,361,-7.170707,62.151825,2020-01,2020-01-01,-72,621,231020000
3,1845,2020-01-01 00:29:20+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,0.944855,380,-7.170737,62.151810,2020-01,2020-01-01,-72,621,231020000
4,1845,2020-01-01 00:35:41+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2.050594,670,-7.170740,62.151798,2020-01,2020-01-01,-72,621,231020000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43636,1694,2020-12-31 23:19:47+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,3.001000,1089,-7.170712,62.151798,2020-12,2020-12-31,-72,621,231020000
43637,1694,2020-12-31 23:31:55+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,0.965000,728,-7.170707,62.151791,2020-12,2020-12-31,-72,621,231020000
43638,1694,2020-12-31 23:37:57+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1.722000,362,-7.170685,62.151802,2020-12,2020-12-31,-72,621,231020000
43639,1694,2020-12-31 23:44:05+00:00,Faeroe Islands,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2.513000,368,-7.170698,62.151825,2020-12,2020-12-31,-72,621,231020000


,date,cell_ll_lat,cell_ll_lon,mmsi,hours,fishing_hours,month,grid_lon,grid_lat,shipid
0,2020-01-01,62.1,-7.2,231020000,22.9361,0.0,2020-01,-72,621,1845
2,2020-01-03,60.0,-3.2,231020000,0.3061,0.0,2020-01,-32,600,1845
3,2020-01-03,60.0,-3.1,231020000,0.5269,0.0,2020-01,-31,600,1845
4,2020-01-03,60.0,-2.7,231020000,0.5338,0.0,2020-01,-27,600,1845
5,2020-01-03,60.0,-3.0,231020000,1.0888,0.0,2020-01,-30,600,1845
...,...,...,...,...,...,...,...,...,...,...
1137,2020-12-07,61.4,-5.9,231020000,0.3977,0.0,2020-12,-59,614,1694
1138,2020-12-07,61.4,-6.0,231020000,0.3855,0.0,2020-12,-60,614,1694
1141,2020-12-07,61.5,-6.1,231020000,0.2841,0.0,2020-12,-61,615,1694
1143,2020-12-07,61.6,-6.2,231020000,0.1322,0.0,2020-12,-62,616,1694


In [34]:
res = 0.1

# Plot ASTD
if origin:
    fig = tb.plot_ship_tracks(
        df1,
        track_ids=[mmsi],
        color_by="track_id",
        show_points=True,
        show_start_end=True,
        map_style="open-street-map",
        zoom=4,
    )

else:
    fig = go.Figure()
    lons_all = []
    lats_all = []

    for _, row in df1.iterrows():
        lon0 = row[col_lon_gfw] * res
        lat0 = row[col_lat_gfw] * res

        lons_all += [lon0, lon0+0.1, lon0+0.1, lon0, lon0, None]
        lats_all += [lat0, lat0, lat0+0.1, lat0+0.1, lat0, None]

    fig.add_trace(go.Scattermap(
        lon=lons_all,
        lat=lats_all,
        mode="lines",
        line=dict(color="blue", width=2),
        name="Cells df1"
    ))

#GFW plot grids
lons_all = []
lats_all = []

for _, row in df2.iterrows():
    lon0 = row[col_lon_gfw] * res
    lat0 = row[col_lat_gfw] * res

    lons_all += [lon0, lon0+0.1, lon0+0.1, lon0, lon0, None]
    lats_all += [lat0, lat0, lat0+0.1, lat0+0.1, lat0, None]


fig.add_trace(go.Scattermap(
    lon=lons_all,
    lat=lats_all,
    mode="lines",
    line=dict(color="red", width=1),
    name="Cells df2"
))

fig.update_layout(
    map=dict(
        style="open-street-map",
        zoom=4,
        center=dict(
            lat=np.average([lat for lat in lats_all if lat is not None]),
            lon=np.average([lon for lon in lons_all if lon is not None])
        )
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    title="Trajectory + Grid 0.1°×0.1°"
)

# tb.export_figure(fig, "output.html")

fig.show()